In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split

class GeneradorSegmentacionTFG(Sequence):
    def __init__(self, carpeta_parches, lista_archivos, batch_size=64, shuffle=True):
        self.carpeta = carpeta_parches
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.archivos_X = lista_archivos
        self.num_clases = 19
        self.indices = np.arange(len(self.archivos_X))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.archivos_X) / self.batch_size))

    def __getitem__(self, index):
        indices_batch = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_X = []
        batch_Y = []
        
        for idx in indices_batch:
            nom_X = self.archivos_X[idx]
            nom_Y = nom_X.replace('_X.npy', '_Y.npy')
            
            parche_X = np.load(os.path.join(self.carpeta, nom_X)).astype(np.float32)
            parche_Y = np.load(os.path.join(self.carpeta, nom_Y)).astype(np.int32)
            
            if parche_X.shape[0] == 4:
                parche_X = np.transpose(parche_X, (1, 2, 0))
            
            parche_X = parche_X[:48, :48, :]
            parche_Y = parche_Y[:48, :48]
            
            for c in range(parche_X.shape[-1]):
                min_val = np.min(parche_X[:,:,c])
                max_val = np.max(parche_X[:,:,c])
                if max_val > min_val:
                    parche_X[:,:,c] = (parche_X[:,:,c] - min_val) / (max_val - min_val)
                else:
                    parche_X[:,:,c] = 0.0
            
            mask_one_hot = tf.one_hot(parche_Y, depth=self.num_clases).numpy()
            
            batch_X.append(parche_X)
            batch_Y.append(mask_one_hot)
            
        if len(batch_X) == 0:
            return (np.zeros((1, 48, 48, 4), dtype=np.float32), 
                    np.zeros((1, 48, 48, self.num_clases), dtype=np.float32))
            
        return np.array(batch_X), np.array(batch_Y)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

carpeta_dataset = r"D:\TFG\dataset_parches_50x50"
todos_los_X_total = sorted([f for f in os.listdir(carpeta_dataset) if f.endswith('_X.npy')])

np.random.seed(4215) 
np.random.shuffle(todos_los_X_total)
archivos_prueba = todos_los_X_total[:50000]

archivos_train, archivos_val = train_test_split(archivos_prueba, test_size=0.2, random_state=42)

generador_train = GeneradorSegmentacionTFG(carpeta_dataset, archivos_train, batch_size=64, shuffle=True)
generador_val = GeneradorSegmentacionTFG(carpeta_dataset, archivos_val, batch_size=64, shuffle=False)

def build_unet_autoencoder(input_shape, num_classes):
    entradas = layers.Input(shape=input_shape)
    
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(entradas)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    bottleneck = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(p2)
    bottleneck = layers.BatchNormalization()(bottleneck)
    
    u3 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(bottleneck)
    merge3 = layers.concatenate([u3, c2]) 
    c3 = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(merge3)
    
    u4 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c3)
    merge4 = layers.concatenate([u4, c1]) 
    c4 = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(merge4)
    
    salidas = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c4)
    
    return models.Model(inputs=entradas, outputs=salidas)

model = build_unet_autoencoder((48, 48, 4), 19)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3, clipnorm=0.5),
    loss='categorical_crossentropy', 
    metrics=['accuracy'] 
)

mis_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

history = model.fit(
    generador_train,
    validation_data=generador_val,
    epochs=40,
    callbacks=mis_callbacks
)

Epoch 1/40
503/625 [=======================>......] - ETA: 1:53 - loss: 36323.2461 - accuracy: 0.1812